# 01 - Speech Commands dataset exploration

This notebook downloads and inspects Google Speech Commands v0.02. It verifies the official split lists, checks the audio files, and explores the ten core command labels plus an `unknown` group. Background noise will be sampled into one-second `silence` examples during preprocessing.

**Do not upload the dataset archive or extracted WAV files to GitHub.**

## 1. Project paths and reproducibility

Run this notebook from the repository root. The download is about 2.3 GB and extraction needs additional disk space. Make sure you have a stable connection and enough free storage before running the download cell.

In [ ]:
from pathlib import Path
import hashlib
import tarfile
import requests
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
ARCHIVE_PATH = DATA_RAW / "speech_commands_v0.02.tar.gz"
EXTRACTED_DIR = DATA_RAW / "speech_commands_v0.02"
DATA_RAW.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
CLIP_SECONDS = 1
CORE_COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_RAW.resolve())

## 2. Download the dataset

Run this cell only when you are ready to download the full archive. It supports resuming an interrupted download only if the server accepts HTTP range requests; otherwise, delete the partial archive and rerun.

In [ ]:
def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Archive already exists: {destination} ({destination.stat().st_size / (1024**3):.2f} GB)")
        return

    response = requests.get(url, stream=True, timeout=(30, 120))
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    temporary_path = destination.with_suffix(destination.suffix + ".part")

    with open(temporary_path, "wb") as file_handle, tqdm(
        total=total, unit="B", unit_scale=True, desc="Downloading Speech Commands"
    ) as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file_handle.write(chunk)
                progress.update(len(chunk))

    temporary_path.replace(destination)
    print("Download complete:", destination)

download_file(DATASET_URL, ARCHIVE_PATH)

## 3. Extract the archive

The extraction step is skipped if the expected split-list file already exists. The archive contains the WAV folders and the official validation/testing file lists.

In [ ]:
if not (EXTRACTED_DIR / "validation_list.txt").exists():
    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(EXTRACTED_DIR, filter="data")
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

print("Validation list:", (EXTRACTED_DIR / "validation_list.txt").exists())
print("Testing list:", (EXTRACTED_DIR / "testing_list.txt").exists())

## 4. Build a file index and reproduce the official splits

The official lists determine validation and test membership. Remaining labelled WAV files are assigned to training. Speaker IDs are retained so we can audit the split and avoid accidental speaker leakage.

In [ ]:
validation_list = set((EXTRACTED_DIR / "validation_list.txt").read_text().splitlines())
testing_list = set((EXTRACTED_DIR / "testing_list.txt").read_text().splitlines())

records = []
for wav_path in EXTRACTED_DIR.rglob("*.wav"):
    relative_path = wav_path.relative_to(EXTRACTED_DIR).as_posix()
    parent_label = wav_path.parent.name.lower()
    if parent_label == "_background_noise_":
        continue
    if relative_path in validation_list:
        split = "validation"
    elif relative_path in testing_list:
        split = "test"
    else:
        split = "train"
    speaker_id = wav_path.stem.split("_nohash_")[0]
    target = parent_label if parent_label in CORE_COMMANDS else "unknown"
    records.append({
        "path": str(wav_path),
        "relative_path": relative_path,
        "source_label": parent_label,
        "target": target,
        "speaker_id": speaker_id,
        "split": split,
    })

audio_index = pd.DataFrame(records)
print("Indexed WAV files:", len(audio_index))
display(audio_index.head())
display(audio_index.groupby(["split", "target"]).size().unstack(fill_value=0))

## 5. Inspect class balance

The ten supported commands remain separate classes. All other spoken words are grouped as `unknown`. We will create a `silence` class from one-second segments of the background-noise recordings during preprocessing.

In [ ]:
class_counts = audio_index.groupby(["split", "target"]).size().unstack(fill_value=0)
ax = class_counts.T.plot(kind="bar", figsize=(13, 5))
ax.set_title("Speech Commands class counts by split")
ax.set_xlabel("Target class")
ax.set_ylabel("Number of WAV files")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Distinct source word folders:", audio_index["source_label"].nunique())
print("Target classes currently indexed:", sorted(audio_index["target"].unique()))
print("Background-noise recordings:", len(list((EXTRACTED_DIR / "_background_noise_").glob("*.wav"))))

## 6. Check sample rate, duration and channel count

This checks a reproducible sample of files rather than opening every audio file. The original recordings are expected to be short, mono audio clips sampled at 16 kHz.

In [ ]:
sample_size = min(500, len(audio_index))
audio_sample = audio_index.sample(sample_size, random_state=42)
audio_metadata = []
for row in audio_sample.itertuples(index=False):
    info = sf.info(row.path)
    audio_metadata.append({
        "sample_rate": info.samplerate,
        "duration_seconds": info.duration,
        "channels": info.channels,
        "subtype": info.subtype,
    })
metadata_df = pd.DataFrame(audio_metadata)
display(metadata_df.describe(include="all"))
print("Sample-rate values:", sorted(metadata_df["sample_rate"].unique()))
print("Channel-count values:", sorted(metadata_df["channels"].unique()))

## 7. Listen to and visualise one audio example

The waveform gives a view of amplitude over time. The spectrogram shows how the signal's frequency content changes over time. These are useful first checks before extracting MFCC features.

In [ ]:
example_row = audio_index[audio_index["target"] == "yes"].iloc[0]
signal, sr = librosa.load(example_row.path, sr=SAMPLE_RATE, mono=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
librosa.display.waveshow(signal, sr=sr, ax=axes[0])
axes[0].set_title(f"Waveform: {example_row.source_label} ({example_row.split})")
axes[0].set_xlabel("Time (seconds)")
D = librosa.amplitude_to_db(np.abs(librosa.stft(signal)), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz", ax=axes[1])
axes[1].set_title("Log-frequency spectrogram")
fig.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
plt.tight_layout()
plt.show()

print("File:", example_row.relative_path)
print("Sample rate:", sr)
print("Duration:", round(len(signal) / sr, 3), "seconds")

## 8. Save the metadata index

The index is small compared with the audio and can be regenerated at any time. It is saved locally under `data/processed/` and should not be committed if it contains machine-specific absolute paths.

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
index_output = PROCESSED_DIR / "audio_index.csv"
audio_index.to_csv(index_output, index=False)
print("Saved:", index_output.resolve())

print("\nDataset exploration complete. Review the class counts, split sizes and audio metadata before proceeding to preprocessing.")